In [1]:
! pip install pypdf
! pip install langchain-pdf


from langchain_community.document_loaders import PyPDFLoader


C:\Users\Asus\AppData\Local\Temp\ipykernel_9508\2217046609.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
# Load a single PDF
loader = PyPDFLoader(r"C:\Users\Asus\Documents\Oulu Research Intern\ClimateIQ\Data\raw\IPCC_AR6_WGI_SPM.pdf")

# Load the document
documents = loader.load()

print(f"Total pages loaded: {len(documents)}")
print(f"\n--- FIRST PAGE ---")
print(f"Content preview: {documents[0].page_content[:500]}")
print(f"\nMetadata: {documents[0].metadata}")

Total pages loaded: 32

--- FIRST PAGE ---
Content preview: Summary for  
Policymakers

Metadata: {'producer': 'Adobe PDF Library 16.0.3', 'creator': 'Adobe InDesign 17.0 (Windows)', 'creationdate': '2022-05-24T12:27:37+02:00', 'author': 'IPCC AR6 Working Group I', 'moddate': '2022-05-24T12:37:29+02:00', 'title': 'Summary for Policymakers', 'trapped': '/False', 'source': 'C:\\Users\\Asus\\Documents\\Oulu Research Intern\\ClimateIQ\\Data\\raw\\IPCC_AR6_WGI_SPM.pdf', 'total_pages': 32, 'page': 0, 'page_label': '1'}


In [3]:
# Look at pages 2, 3, and 4
for i in [1, 2, 3]:
    print(f"\n--- PAGE {i+1} ---")
    print(f"Content preview: {documents[i].page_content[:300]}")
    print(f"Page number: {documents[i].metadata['page']}")


--- PAGE 2 ---
Content preview: 
Page number: 1

--- PAGE 3 ---
Content preview: SPM
3
Drafting Authors:
Richard P . Allan (United Kingdom), Paola A. Arias (Colombia), Sophie Berger (France/Belgium), Josep G. 
Canadell (Australia), Christophe Cassou (France), Deliang Chen (Sweden), Annalisa Cherchi (Italy), Sarah 
L. Connors (France/United Kingdom), Erika Coppola (Italy), Faye A
Page number: 2

--- PAGE 4 ---
Content preview: 4
SPM
Summary for Policymakers
Introduction
1  Decision IPCC/XLVI-2.
2  The three Special Reports are: Global Warming of 1.5°C: An IPCC Special Report on the impacts of global warming of 1.5°C above pre-industrial levels and related global greenhouse 
gas emission pathways, in the context of strengt
Page number: 3


In [4]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

# Load all PDFs from Data/raw folder
loader = DirectoryLoader(
    r"C:\Users\Asus\Documents\Oulu Research Intern\ClimateIQ\Data\raw",
    glob="*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True,
    use_multithreading=True
)

# Use lazy_load to process one document at a time
all_documents = []
for doc in loader.lazy_load():
    all_documents.append(doc)

print(f"\nTotal pages loaded across all documents: {len(all_documents)}")

# Summary per document
from collections import defaultdict
doc_pages = defaultdict(int)
for doc in all_documents:
    source = doc.metadata['source'].split('\\')[-1]
    doc_pages[source] += 1

print(f"\nPages per document:")
for filename, count in doc_pages.items():
    print(f"  {filename}: {count} pages")

  0%|          | 0/6 [00:00<?, ?it/s]

100%|██████████| 6/6 [02:29<00:00, 24.86s/it]


Total pages loaded across all documents: 420

Pages per document:
  IPCC_AR6_WGII_SummaryForPolicymakers.pdf: 34 pages
  IPCC_AR6_WGIII_SummaryForPolicymakers.pdf: 56 pages
  IPCC_AR6_WGI_SPM.pdf: 32 pages
  IPCC_AR6_WGIII_TechnicalSummary.pdf: 102 pages
  IPCC_AR6_WGII_TechnicalSummary.pdf: 84 pages
  IPCC_AR6_WGI_TS.pdf: 112 pages


In [5]:
pip install langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create a splitter
# chunk_size is in characters, not words. 500 words ≈ 2500 characters (avg 5 chars/word)
# Use chunk_size=2500 and chunk_overlap=250 (10% overlap)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2500,
    chunk_overlap=250
)

# Split all documents into chunks
chunks = text_splitter.split_documents(all_documents)

print(f"Total chunks created: {len(chunks)}")
print(f"\n--- FIRST CHUNK ---")
print(chunks[0].page_content)
print(f"\nMetadata: {chunks[0].metadata}")

Total chunks created: 1007

--- FIRST CHUNK ---
Summary for  
Policymakers

Metadata: {'producer': 'Adobe PDF Library 16.0.5', 'creator': 'Adobe InDesign 17.1 (Windows)', 'creationdate': '2022-07-29T16:15:21+02:00', 'moddate': '2022-07-29T16:15:30+02:00', 'trapped': '/False', 'source': 'C:\\Users\\Asus\\Documents\\Oulu Research Intern\\ClimateIQ\\Data\\raw\\IPCC_AR6_WGII_SummaryForPolicymakers.pdf', 'total_pages': 34, 'page': 0, 'page_label': '1'}


In [7]:
avg = sum(len(c.page_content) for c in chunks) // len(chunks)
print(f"Average chunk size: {avg} characters")
print(f"Shortest chunk: {min(len(c.page_content) for c in chunks)} characters")
print(f"Longest chunk: {max(len(c.page_content) for c in chunks)} characters")

Average chunk size: 2008 characters
Shortest chunk: 2 characters
Longest chunk: 2500 characters


In [8]:
# Filter out chunks that are too short to be useful
min_chunk_size = 100  # minimum 100 characters

filtered_chunks = [chunk for chunk in chunks if len(chunk.page_content) >= min_chunk_size]

print(f"Chunks before filtering: {len(chunks)}")
print(f"Chunks after filtering: {len(filtered_chunks)}")
print(f"Removed {len(chunks) - len(filtered_chunks)} useless chunks")


Chunks before filtering: 1007
Chunks after filtering: 1000
Removed 7 useless chunks


In [9]:
# Create chunks at three different sizes
chunk_configs = [
    {"size": 1000, "overlap": 100},
    {"size": 2500, "overlap": 250},
    {"size": 4000, "overlap": 400}
]

all_chunk_sets = {}

for config in chunk_configs:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config["size"],
        chunk_overlap=config["overlap"],
        length_function=len,
        separators=["\n\n", "\n", ".", " ", ""]
    )
    
    chunks = splitter.split_documents(all_documents)
    
    # Filter useless chunks
    filtered = [c for c in chunks if len(c.page_content) >= 100]
    
    all_chunk_sets[config["size"]] = filtered
    
    print(f"Chunk size {config['size']}: {len(filtered)} chunks")
    

Chunk size 1000: 2247 chunks
Chunk size 2500: 1000 chunks
Chunk size 4000: 671 chunks


In [10]:
# Check a sample chunk from each size
for size in [1000, 2500, 4000]:
    sample = all_chunk_sets[size][10]
    print(f"\n--- CHUNK SIZE {size} --- SAMPLE ---")
    print(f"Characters: {len(sample.page_content)}")
    print(f"Content: {sample.page_content[:200]}")
    print(f"Source: {sample.metadata['source'].split(chr(92))[-1]}")
    print(f"Page: {sample.metadata['page']}")
    


--- CHUNK SIZE 1000 --- SAMPLE ---
Characters: 982
Content: 5
SPM
Summary for Policymakers
A: Introduction
This Summary for Policymakers (SPM) presents key findings of the Working Group II (WGII) contribution to the Sixth Assessment Report (AR6) of 
the IPCC1.
Source: IPCC_AR6_WGII_SummaryForPolicymakers.pdf
Page: 4

--- CHUNK SIZE 2500 --- SAMPLE ---
Characters: 2444
Content: 7
SPM
Summary for Policymakers
and/ or transformational. The latter changes the fundamental attributes of a social-ecological system in anticipation of climate change and its 
impacts. Adaptation is s
Source: IPCC_AR6_WGII_SummaryForPolicymakers.pdf
Page: 6

--- CHUNK SIZE 4000 --- SAMPLE ---
Characters: 3880
Content: 9
SPM
Summary for Policymakers
Observed Impacts from Climate Change
28 Attribution is defined as the process of evaluating the relative contributions of multiple causal factors to a change or event wi
Source: IPCC_AR6_WGII_SummaryForPolicymakers.pdf
Page: 8


In [11]:
from langchain_huggingface import HuggingFaceEmbeddings

# Load embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

# Test on one chunk
test_chunk = all_chunk_sets[2500][0].page_content
test_vector = embeddings.embed_query(test_chunk)

print(f"Embedding model loaded successfully")
print(f"Vector length: {len(test_vector)}")
print(f"First 5 values: {test_vector[:5]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully
Vector length: 384
First 5 values: [-0.012812897562980652, -0.009268308989703655, -0.044607967138290405, 0.017154691740870476, 0.04744896665215492]


In [12]:
import time

# Time how long one chunk takes to embed
start = time.time()
test_vector = embeddings.embed_query(all_chunk_sets[2500][0].page_content)
end = time.time()

time_per_chunk = end - start
total_chunks = sum(len(all_chunk_sets[size]) for size in [1000, 2500, 4000])

print(f"Time per chunk: {time_per_chunk:.2f} seconds")
print(f"Total chunks to embed: {total_chunks}")
print(f"Estimated total time: {(time_per_chunk * total_chunks) / 60:.1f} minutes")

Time per chunk: 0.08 seconds
Total chunks to embed: 3918
Estimated total time: 4.9 minutes


In [13]:
from langchain_chroma import Chroma
import os

# Path where ChromaDB will save data on your laptop
chroma_path = r"C:\Users\Asus\Documents\Oulu Research Intern\ClimateIQ\Data\chroma_db"

# Create one collection per chunk size
for size in [1000, 2500, 4000]:
    print(f"Creating collection for chunk size {size}...")
    
    collection_name = f"climate_chunks_{size}"
    
    vectorstore = Chroma.from_documents(
        documents=all_chunk_sets[size],
        embedding=embeddings,
        collection_name=collection_name,
        persist_directory=chroma_path
    )
    
    print(f"Collection {collection_name} created with {len(all_chunk_sets[size])} chunks")

print(f"\nAll collections created and saved to {chroma_path}")

Creating collection for chunk size 1000...
Collection climate_chunks_1000 created with 2247 chunks
Creating collection for chunk size 2500...
Collection climate_chunks_2500 created with 1000 chunks
Creating collection for chunk size 4000...
Collection climate_chunks_4000 created with 671 chunks

All collections created and saved to C:\Users\Asus\Documents\Oulu Research Intern\ClimateIQ\Data\chroma_db


In [14]:
# Load an existing collection and do a test search
test_store = Chroma(
    collection_name="climate_chunks_2500",
    embedding_function=embeddings,
    persist_directory=chroma_path
)

# Search for something
query = "What is the current global temperature rise?"
results = test_store.similarity_search(query, k=3)

print(f"Query: {query}")
print(f"Top 3 results:\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Source: {doc.metadata['source'].split(chr(92))[-1]}")
    print(f"Page: {doc.metadata['page']}")
    print(f"Content: {doc.page_content[:500]}")
    print()

Query: What is the current global temperature rise?
Top 3 results:

--- Result 1 ---
Source: IPCC_AR6_WGI_SPM.pdf
Page: 4
Content: 5
SPM
Summary for Policymakers
A.1.2   Each of the last four decades has been successively warmer than any decade that preceded it since 1850. Global 
surface temperature8 in the first two decades of the 21st century (2001–2020) was 0.99 [0.84 to 1.10] °C higher than 
1850–1900.9 Global surface temperature was 1.09 [0.95 to 1.20] °C higher in 2011–2020 than 1850–1900, with larger 
increases over land (1.59 [1.34 to 1.83] °C) than over the ocean (0.88 [0.68 to 1.01] °C). The estimated increase in

--- Result 2 ---
Source: IPCC_AR6_WGI_SPM.pdf
Page: 4
Content: 5
SPM
Summary for Policymakers
A.1.2   Each of the last four decades has been successively warmer than any decade that preceded it since 1850. Global 
surface temperature8 in the first two decades of the 21st century (2001–2020) was 0.99 [0.84 to 1.10] °C higher than 
1850–1900.9 Global surface temperat

In [15]:
pip install langchain langchain-groq

Note: you may need to restart the kernel to use updated packages.


In [16]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv

load_dotenv()

# Load the LLM
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

# Load your vector store
vectorstore = Chroma(
    collection_name="climate_chunks_2500",
    embedding_function=embeddings,
    persist_directory=chroma_path
)

# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Create prompt template
prompt = PromptTemplate.from_template("""You are ClimateIQ, a climate science assistant.
Answer the question using ONLY the context provided below.
If the answer is not in the context, say "I don't have enough information to answer this."
Always mention which document your answer comes from.

Context:
{context}

Question: {question}

Answer:""")

# Helper function to format retrieved docs
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build modern RAG chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Ask a question
query = "What is the current global temperature rise?"
response = rag_chain.invoke(query)

print(f"Question: {query}")
print(f"\nAnswer: {response}")

Question: What is the current global temperature rise?

Answer: The current global temperature rise is 0.99 [0.84 to 1.10] °C higher than 1850–1900, and 1.09 [0.95 to 1.20] °C higher in 2011–2020 than 1850–1900. (Source: SPM, A.1.2)


In [17]:
test_questions = [
    "What are the main causes of climate change?",
    "What will happen to sea levels by 2100?"
]

for question in test_questions:
    print(f"\nQuestion: {question}")
    response = rag_chain.invoke(question)
    print(f"Answer: {response}")
    print("-" * 50)


Question: What are the main causes of climate change?
Answer: I don't have enough information to answer this.
--------------------------------------------------

Question: What will happen to sea levels by 2100?
Answer: Sea level will increase an additional 30 cm to 1 m or more by 2100, depending on future emissions. 

Document: 89, Technical Summary, TS
--------------------------------------------------


In [18]:
# Check what was actually retrieved for that question
query = "What are the main causes of climate change?"
docs = retriever.invoke(query)

print(f"Retrieved chunks for: '{query}'\n")
for i, doc in enumerate(docs):
    print(f"--- Chunk {i+1} ---")
    print(f"Source: {doc.metadata['source'].split(chr(92))[-1]}")
    print(f"Content: {doc.page_content[:300]}")
    print()

Retrieved chunks for: 'What are the main causes of climate change?'

--- Chunk 1 ---
Source: IPCC_AR6_WGII_TechnicalSummary.pdf
Content: both direct drivers (e.g., destruction of homes by tropical cyclones) and 
indirect drivers (e.g., rural income losses during prolonged droughts) 
of involuntary migration and displacement (very high confidence). 
The largest absolute number of people displaced by extreme weather 
each year occurs i

--- Chunk 2 ---
Source: IPCC_AR6_WGII_TechnicalSummary.pdf
Content: both direct drivers (e.g., destruction of homes by tropical cyclones) and 
indirect drivers (e.g., rural income losses during prolonged droughts) 
of involuntary migration and displacement (very high confidence). 
The largest absolute number of people displaced by extreme weather 
each year occurs i

--- Chunk 3 ---
Source: IPCC_AR6_WGII_TechnicalSummary.pdf
Content: both direct drivers (e.g., destruction of homes by tropical cyclones) and 
indirect drivers (e.g., rural income losses duri

In [19]:
query = "Human influence greenhouse gas emissions warming"
docs = retriever.invoke(query)

print(f"Retrieved chunks:\n")
for i, doc in enumerate(docs):
    print(f"--- Chunk {i+1} ---")
    print(f"Source: {doc.metadata['source'].split(chr(92))[-1]}")
    print(f"Content: {doc.page_content[:300]}")
    print()

Retrieved chunks:

--- Chunk 1 ---
Source: IPCC_AR6_WGI_TS.pdf
Content: levels and estimates of remaining carbon budgets are updated 
accordingly. (Section TS.1.2, Cross-Section Box TS.1)
• Paleoclimate evidence:  The AR5 assessed that many of the 
changes observed since the 1950s are unprecedented over 
decades to millennia. Updated paleoclimate evidence strengthens 
t

--- Chunk 2 ---
Source: IPCC_AR6_WGI_TS.pdf
Content: levels and estimates of remaining carbon budgets are updated 
accordingly. (Section TS.1.2, Cross-Section Box TS.1)
• Paleoclimate evidence:  The AR5 assessed that many of the 
changes observed since the 1950s are unprecedented over 
decades to millennia. Updated paleoclimate evidence strengthens 
t

--- Chunk 3 ---
Source: IPCC_AR6_WGI_TS.pdf
Content: levels and estimates of remaining carbon budgets are updated 
accordingly. (Section TS.1.2, Cross-Section Box TS.1)
• Paleoclimate evidence:  The AR5 assessed that many of the 
changes observed since the 1950s are unpr

In [20]:
query = "What is the human influence on climate change and greenhouse gas emissions?"
response = rag_chain.invoke(query)
print(f"Question: {query}")
print(f"\nAnswer: {response}")

Question: What is the human influence on climate change and greenhouse gas emissions?

Answer: Human influence on the climate system refers to human-driven activities that lead to changes in the climate system due to perturbations of Earth’s energy budget (also called anthropogenic forcing). Human influence results from emissions of greenhouse gases, aerosols and tropospheric ozone precursors, ozone-depleting substances, and land-use change. (Document: Section 9)


In [21]:
from langchain_community.retrievers import BM25Retriever

# Build BM25 retriever using 2500 character chunks
bm25_retriever = BM25Retriever.from_documents(
    all_chunk_sets[2500],
    k=3
)

# Test it on the question that failed with dense retrieval
query = "What are the main causes of climate change?"
results = bm25_retriever.invoke(query)

print(f"Query: {query}")
print(f"\nBM25 retrieved {len(results)} chunks:\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Source: {doc.metadata['source'].split(chr(92))[-1]}")
    print(f"Page: {doc.metadata['page']}")
    print(f"Content: {doc.page_content[:300]}")
    print()

Query: What are the main causes of climate change?

BM25 retrieved 3 chunks:

--- Result 1 ---
Source: IPCC_AR6_WGIII_TechnicalSummary.pdf
Page: 30
Content: corresponds to reductions of 34–60% by 2030 and 73–98% by 2050 
relative to 2019 levels. {3.3}
Box TS.5 | Illustrative Mitigation Pathways (IMPs), and Shared Socio-economic Pathways (SSPs)
The Illustrative Mitigation Pathways (IMPs)
The over 2500 model-based pathways submitted to the AR6 scenarios d

--- Result 2 ---
Source: IPCC_AR6_WGI_TS.pdf
Page: 74
Content: are important data sources, and discarding models that 
fundamentally misrepresent relevant processes improves 
the credibility of ensemble information related to these 
processes. A key methodology is distillation – combining 
lines of evidence and accounting for stakeholder context 
and values – w

--- Result 3 ---
Source: IPCC_AR6_WGI_TS.pdf
Page: 8
Content: 41
Technical Summary
TS
Interactive Atlas allows users to interact in a flexible manner through 
maps, time series

In [22]:
# Search specifically for the right content
query2 = "greenhouse gas concentrations caused by human activities"
results2 = bm25_retriever.invoke(query2)

print(f"Query: {query2}\n")
for i, doc in enumerate(results2):
    print(f"--- Result {i+1} ---")
    print(f"Source: {doc.metadata['source'].split(chr(92))[-1]}")
    print(f"Content: {doc.page_content[:300]}")
    print()

Query: greenhouse gas concentrations caused by human activities

--- Result 1 ---
Source: IPCC_AR6_WGI_SPM.pdf
Content: Box 9.1} (Figure SPM.1, Figure SPM.2)
A.1.1  Observed increases in well-mixed greenhouse gas (GHG) concentrations since around 1750 are unequivocally caused 
by human activities. Since 2011 (measurements reported in AR5), concentrations have continued to increase in the 
atmosphere, reaching annual 

--- Result 2 ---
Source: IPCC_AR6_WGI_SPM.pdf
Content: net zero GHG emissions, where metric-weighted anthropogenic GHG emissions equal metric-weighted anthropogenic 
GHG removals. For a given GHG emissions pathway, the pathways of individual GHGs determine the resulting climate 
response,46 whereas the choice of emissions metric 47 used to calculate agg

--- Result 3 ---
Source: IPCC_AR6_WGI_TS.pdf
Content: A related term – remaining carbon budget – is used to describe the total net amount of CO 2 that could be released in the future by 
human activities while keeping glo

In [23]:
import langchain
import langchain_community
print(f"LangChain version: {langchain.__version__}")
print(f"LangChain Community version: {langchain_community.__version__}")

LangChain version: 1.3.9
LangChain Community version: 0.4.2


In [24]:
from langchain_classic.retrievers import EnsembleRetriever

In [25]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

# BM25 retriever
bm25_retriever = BM25Retriever.from_documents(
    all_chunk_sets[2500],
    k=1
)

# Dense retriever
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Hybrid — combine both with equal weight
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.5, 0.5]
)

# Test on the question that failed both retrievers individually
query = "What are the main causes of climate change?"
results = ensemble_retriever.invoke(query)[:3]

print(f"Query: {query}\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Source: {doc.metadata['source'].split(chr(92))[-1]}")
    print(f"Page: {doc.metadata['page']}")
    print(f"Content: {doc.page_content[:500]}")
    print()

Query: What are the main causes of climate change?

--- Result 1 ---
Source: IPCC_AR6_WGII_TechnicalSummary.pdf
Page: 17
Content: both direct drivers (e.g., destruction of homes by tropical cyclones) and 
indirect drivers (e.g., rural income losses during prolonged droughts) 
of involuntary migration and displacement (very high confidence). 
The largest absolute number of people displaced by extreme weather 
each year occurs in Asia (South, Southeast and East), followed by 
sub-Saharan Africa, but small island states in the Caribbean and 
South Pacific are disproportionately affected relative to their small 
population siz

--- Result 2 ---
Source: IPCC_AR6_WGIII_TechnicalSummary.pdf
Page: 30
Content: corresponds to reductions of 34–60% by 2030 and 73–98% by 2050 
relative to 2019 levels. {3.3}
Box TS.5 | Illustrative Mitigation Pathways (IMPs), and Shared Socio-economic Pathways (SSPs)
The Illustrative Mitigation Pathways (IMPs)
The over 2500 model-based pathways submitted to the AR6 

In [26]:
# Search with the exact words from the correct chunk
query = "unequivocally caused by human activities greenhouse gas"
results = vectorstore.similarity_search(query, k=5)

print(f"Searching for exact IPCC language:\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Source: {doc.metadata['source'].split(chr(92))[-1]}")
    print(f"Page: {doc.metadata['page']}")
    print(f"Content: {doc.page_content[:400]}")
    print()

Searching for exact IPCC language:

--- Result 1 ---
Source: IPCC_AR6_WGI_SPM.pdf
Page: 3
Content: Box 9.1} (Figure SPM.1, Figure SPM.2)
A.1.1  Observed increases in well-mixed greenhouse gas (GHG) concentrations since around 1750 are unequivocally caused 
by human activities. Since 2011 (measurements reported in AR5), concentrations have continued to increase in the 
atmosphere, reaching annual averages of 410 parts per million (ppm) for carbon dioxide (CO 2), 1866 parts per billion 
(ppb) for

--- Result 2 ---
Source: IPCC_AR6_WGI_SPM.pdf
Page: 3
Content: Box 9.1} (Figure SPM.1, Figure SPM.2)
A.1.1  Observed increases in well-mixed greenhouse gas (GHG) concentrations since around 1750 are unequivocally caused 
by human activities. Since 2011 (measurements reported in AR5), concentrations have continued to increase in the 
atmosphere, reaching annual averages of 410 parts per million (ppm) for carbon dioxide (CO 2), 1866 parts per billion 
(ppb) for

--- Result 3 ---
Source: IPCC_AR6_

In [27]:
print("all_documents" in dir())
print("all_chunk_sets" in dir())
print("embeddings" in dir())
print("vectorstore" in dir())


True
True
True
True


In [1]:
import shutil
import os

chroma_path = r"C:\Users\Asus\Documents\Oulu Research Intern\ClimateIQ\Data\chroma_db"

if os.path.exists(chroma_path):
    shutil.rmtree(chroma_path)
    print("Existing ChromaDB deleted")

os.makedirs(chroma_path)
print("Fresh ChromaDB folder created")

Existing ChromaDB deleted
Fresh ChromaDB folder created


In [2]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_community.retrievers import BM25Retriever
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
import os
import re

load_dotenv()

# Reload embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

chroma_path = r"C:\Users\Asus\Documents\Oulu Research Intern\ClimateIQ\Data\chroma_db"

print("All imports done")

C:\Users\Asus\AppData\Local\Temp\ipykernel_8648\3498268984.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

All imports done


In [3]:
loader = DirectoryLoader(
    r"C:\Users\Asus\Documents\Oulu Research Intern\ClimateIQ\Data\raw",
    glob="*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True
)

all_documents = list(loader.lazy_load())
print(f"Total pages loaded: {len(all_documents)}")

100%|██████████| 6/6 [02:32<00:00, 25.40s/it]

Total pages loaded: 420


In [4]:
def clean_document(text):
    if text is None:
        return ""
    text = re.sub(r'Figure\s+[A-Z]*\s*\d+\.\d+', '', text)
    text = re.sub(r'Box\s+[A-Z]*\s*\d+\.\d+', '', text)
    text = re.sub(r'\b[A-Z]{1,2}\.\d+\.\d+\b', '', text)
    text = re.sub(r'\{[^}]+\}', '', text)
    text = re.sub(r'^\d+$', '', text, flags=re.MULTILINE)
    text = re.sub(r'^[-_]+$', '', text, flags=re.MULTILINE)
    text = re.sub(r' +', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = text.strip()
    return text

for doc in all_documents:
    doc.page_content = clean_document(doc.page_content)

print("Cleaning applied to all documents")

Cleaning applied to all documents


In [5]:
chunk_configs = [
    {"size": 1000, "overlap": 100},
    {"size": 2500, "overlap": 250},
    {"size": 4000, "overlap": 400}
]

all_chunk_sets = {}

for config in chunk_configs:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config["size"],
        chunk_overlap=config["overlap"],
        length_function=len,
        separators=["\n\n", "\n", ".", " ", ""]
    )
    chunks = splitter.split_documents(all_documents)
    filtered = [c for c in chunks if len(c.page_content) >= 100]
    all_chunk_sets[config["size"]] = filtered
    print(f"Chunk size {config['size']}: {len(filtered)} chunks")

Chunk size 1000: 2200 chunks
Chunk size 2500: 1008 chunks
Chunk size 4000: 695 chunks


In [6]:
for size in [1000, 2500, 4000]:
    print(f"Creating collection for chunk size {size}...")
    collection_name = f"climate_chunks_{size}"
    vectorstore = Chroma.from_documents(
        documents=all_chunk_sets[size],
        embedding=embeddings,
        collection_name=collection_name,
        persist_directory=chroma_path
    )
    print(f"Done — {len(all_chunk_sets[size])} chunks stored")

print("\nAll collections rebuilt")

Creating collection for chunk size 1000...
Done — 2200 chunks stored
Creating collection for chunk size 2500...
Done — 1008 chunks stored
Creating collection for chunk size 4000...
Done — 695 chunks stored

All collections rebuilt


In [8]:
vectorstore = Chroma(
    collection_name="climate_chunks_2500",
    embedding_function=embeddings,
    persist_directory=chroma_path
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

query = "What are the main causes of climate change?"
results = retriever.invoke(query)

print(f"Query: {query}\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Source: {doc.metadata['source'].split(chr(92))[-1]}")
    print(f"Page: {doc.metadata['page']}")
    print(f"Content: {doc.page_content[:600]}")
    print()

Query: What are the main causes of climate change?

--- Result 1 ---
Source: IPCC_AR6_WGI_SPM.pdf
Page: 7
Content: A.3 Human-induced climate change is already affecting many weather and climate extremes in every region 
across the globe. Evidence of observed changes in extremes such as heatwaves, heavy precipitation, droughts, 
and tropical cyclones, and, in particular, their attribution to human influence, has strengthened since AR5. 
 
(Figure SPM.3)
 It is virtually certain that hot extremes (including heatwaves) have become more frequent and more intense across most 
land regions since the 1950s, while cold extremes (including cold waves) have become less frequent and less severe, with 
high confidence

--- Result 2 ---
Source: IPCC_AR6_WGII_TechnicalSummary.pdf
Page: 17
Content: TS. The most common climatic drivers for migration and 
displacement are drought, tropical storms and hurricanes, heavy 
rains and floods (high confidence). Extreme climate events act as 
both direct drive

In [9]:
query = "greenhouse gas concentrations caused human activities"
results = retriever.invoke(query)

print(f"Query: {query}\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Source: {doc.metadata['source'].split(chr(92))[-1]}")
    print(f"Page: {doc.metadata['page']}")
    print(f"Content: {doc.page_content[:300]}")
    print()

Query: greenhouse gas concentrations caused human activities

--- Result 1 ---
Source: IPCC_AR6_WGI_SPM.pdf
Page: 3
Content: Observed increases in well-mixed greenhouse gas (GHG) concentrations since around 1750 are unequivocally caused 
by human activities. Since 2011 (measurements reported in AR5), concentrations have continued to increase in the 
atmosphere, reaching annual averages of 410 parts per million (ppm) for c

--- Result 2 ---
Source: IPCC_AR6_WGI_TS.pdf
Page: 11
Content: Technical Summary
TS
Period [2.5°C to 4°C], about 3.2 million years ago, whereas the high 
CO2 emissions scenario SSP5-8.5 leads to temperatures of [6.6°C 
to 14.1°C] by 2300, which overlaps with the Early Eocene Climate 
Optimum [10°C to 18°C], about 50 million years ago. 
Understanding of the clim

--- Result 3 ---
Source: IPCC_AR6_WGI_TS.pdf
Page: 34
Content: (since 1960)
↑
(Scenario-based assessment for 21st century)
 Changes in the Drivers of the Climate System
Since 1750, changes in the drivers of t

In [11]:
from langchain_community.retrievers import BM25Retriever

# Build BM25 retriever on cleaned chunks
bm25_retriever = BM25Retriever.from_documents(
    all_chunk_sets[2500],
    k=3
)

# Test on the failing query
query = "What are the main causes of climate change?"
results = bm25_retriever.invoke(query)

print(f"Query: {query}\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Source: {doc.metadata['source'].split(chr(92))[-1]}")
    print(f"Page: {doc.metadata['page']}")
    print(f"Content: {doc.page_content[:500]}")
    print()

Query: What are the main causes of climate change?

--- Result 1 ---
Source: IPCC_AR6_WGIII_TechnicalSummary.pdf
Page: 30
Content: corresponds to reductions of 34–60% by 2030 and 73–98% by 2050 
relative to 2019 levels. 
Box TS.5 | Illustrative Mitigation Pathways (IMPs), and Shared Socio-economic Pathways (SSPs)
The Illustrative Mitigation Pathways (IMPs)
The over 2500 model-based pathways submitted to the AR6 scenarios database pathways explore different possible evolutions of 
future energy and land use (with and without climate policy) and the consequences for greenhouse gas emissions. 
From the full range of pathways, 

--- Result 2 ---
Source: IPCC_AR6_WGI_TS.pdf
Page: 74
Content: fundamentally misrepresent relevant processes improves 
the credibility of ensemble information related to these 
processes. A key methodology is distillation – combining 
lines of evidence and accounting for stakeholder context 
and values – which helps ensure the information is relevant, 
useful and t

In [14]:
from langchain_classic.retrievers import EnsembleRetriever

dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.5, 0.5]
)

query = "What are the main causes of climate change?"
results = ensemble_retriever.invoke(query)[:3]

print(f"Query: {query}\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Source: {doc.metadata['source'].split(chr(92))[-1]}")
    print(f"Page: {doc.metadata['page']}")
    print(f"Content: {doc.page_content[:500]}")
    print()

Query: What are the main causes of climate change?

--- Result 1 ---
Source: IPCC_AR6_WGIII_TechnicalSummary.pdf
Page: 30
Content: corresponds to reductions of 34–60% by 2030 and 73–98% by 2050 
relative to 2019 levels. 
Box TS.5 | Illustrative Mitigation Pathways (IMPs), and Shared Socio-economic Pathways (SSPs)
The Illustrative Mitigation Pathways (IMPs)
The over 2500 model-based pathways submitted to the AR6 scenarios database pathways explore different possible evolutions of 
future energy and land use (with and without climate policy) and the consequences for greenhouse gas emissions. 
From the full range of pathways, 

--- Result 2 ---
Source: IPCC_AR6_WGI_SPM.pdf
Page: 7
Content: A.3 Human-induced climate change is already affecting many weather and climate extremes in every region 
across the globe. Evidence of observed changes in extremes such as heatwaves, heavy precipitation, droughts, 
and tropical cyclones, and, in particular, their attribution to human influence, has stre

In [16]:
! pip install --upgrade langchain langchain-community


In [18]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_groq import ChatGroq
import os

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

# Build multi query retriever on top of dense retriever
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    llm=llm
)

# Test on the failing query
query = "What are the main causes of climate change?"
results = multi_query_retriever.invoke(query)

print(f"Query: {query}")
print(f"Total chunks retrieved: {len(results)}\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Source: {doc.metadata['source'].split(chr(92))[-1]}")
    print(f"Page: {doc.metadata['page']}")
    print(f"Content: {doc.page_content[:300]}")
    print()

Query: What are the main causes of climate change?
Total chunks retrieved: 10

--- Result 1 ---
Source: IPCC_AR6_WGI_TS.pdf
Page: 78
Content: Technical Summary
TS
Box TS.11 | Climate Services
Climate services involve providing climate information to assist decision-making, for example, about how extreme 
rainfall will change to inform improvements in urban drainage. Since AR5, there has been a significant increase in the 
range and divers

--- Result 2 ---
Source: IPCC_AR6_WGI_TS.pdf
Page: 77
Content: where there is at least medium confidence in an observed decrease in agricultural and ecological drought.
For all regions, Table TS.5 shows a broader range of observed changes besides the ones shown in this figure. Note that Southern South America (SSA) is the only 
region that does not display obse

--- Result 3 ---
Source: IPCC_AR6_WGI_TS.pdf
Page: 74
Content: fundamentally misrepresent relevant processes improves 
the credibility of ensemble information related to these 
processes. A k

In [19]:
# Search directly for A.1.1 content
query = "Observed increases well-mixed greenhouse gas concentrations 1750 unequivocally caused human activities"
results = vectorstore.similarity_search(query, k=3)

print("Direct search for A.1.1 content:\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Source: {doc.metadata['source'].split(chr(92))[-1]}")
    print(f"Page: {doc.metadata['page']}")
    print(f"Content: {doc.page_content[:400]}")
    print()

Direct search for A.1.1 content:

--- Result 1 ---
Source: IPCC_AR6_WGI_SPM.pdf
Page: 3
Content: Observed increases in well-mixed greenhouse gas (GHG) concentrations since around 1750 are unequivocally caused 
by human activities. Since 2011 (measurements reported in AR5), concentrations have continued to increase in the 
atmosphere, reaching annual averages of 410 parts per million (ppm) for carbon dioxide (CO 2), 1866 parts per billion 
(ppb) for methane (CH 4), and 332 ppb for nitrous oxid

--- Result 2 ---
Source: IPCC_AR6_WGI_TS.pdf
Page: 59
Content: Technical Summary
TS
For CO2, CH4, N2O, and chlorofluorocarbons, there is now evidence to 
quantify the effect on ERF of tropospheric adjustments. The assessed 
ERF for a doubling of CO2 compared to 1750 levels (3.9 ± 0.5 Wm–2) is 
larger than in AR5. For CO2, the adjustments include the physiological 
effects on vegetation. The reactive well-mixed greenhouse gases 
(CH4, N2O, and halocarbons) cau

--- Result 3 ---
Source: IPCC_AR6_WG

In [20]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os

load_dotenv()

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

# Query expansion prompt
query_expansion_prompt = PromptTemplate.from_template("""You are a climate science expert familiar with IPCC reports.
Rewrite the following question using precise scientific terminology 
that would appear in IPCC assessment reports.
Return only the rewritten question, nothing else.

Original question: {question}

Rewritten question:""")

# Build query expansion chain
query_expander = query_expansion_prompt | llm | StrOutputParser()

# Test it
original_query = "What are the main causes of climate change?"
expanded_query = query_expander.invoke({"question": original_query})

print(f"Original query: {original_query}")
print(f"Expanded query: {expanded_query}")

Original query: What are the main causes of climate change?
Expanded query: What are the primary anthropogenic and natural drivers of global warming, as identified in the latest available scientific evidence and assessed in accordance with the principles outlined in the IPCC's Framework for the Construction of Scenarios?


In [22]:
# Test retrieval with expanded query
results = vectorstore.similarity_search(expanded_query, k=3)

print(f"Expanded query: {expanded_query}\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Source: {doc.metadata['source'].split(chr(92))[-1]}")
    print(f"Page: {doc.metadata['page']}")
    print(f"Content: {doc.page_content[:600]}")
    print()

Expanded query: What are the primary anthropogenic and natural drivers of global warming, as identified in the latest available scientific evidence and assessed in accordance with the principles outlined in the IPCC's Framework for the Construction of Scenarios?

--- Result 1 ---
Source: IPCC_AR6_WGI_TS.pdf
Page: 10
Content: used throughout this Report is summarized in Cross-Section Box TS.1.
TS1.1 Context of a Changing Climate
This Report assesses new scientific evidence relevant 
for a world whose climate system is rapidly changing, 
overwhelmingly due to human influence. The five IPCC 
assessment cycles since 1990 have comprehensively and 
consistently laid out the rapidly accumulating evidence 
of a changing climate system, with the Fourth Assessment 
Report in 2007 being the first to conclude that warming 
of the climate system is unequivocal. Sustained changes 
have been documented in all major elements of t

--- Result 2 ---
Source: IPCC_AR6_WGIII_TechnicalSummary.pdf
Page: 6
Co

In [23]:
# Full pipeline with query expansion
original_query = "What are the main causes of climate change?"

# Step 1 - expand query
expanded_query = query_expander.invoke({"question": original_query})
print(f"Expanded: {expanded_query}\n")

# Step 2 - retrieve with expanded query
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Step 3 - build RAG chain
prompt = PromptTemplate.from_template("""You are ClimateIQ, a climate science assistant.
Answer the question using ONLY the context provided below.
If the answer is not in the context, say "I don't have enough information."
Always mention which document your answer comes from.

Context:
{context}

Question: {question}

Answer:""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Retrieve using expanded query
retrieved_docs = retriever.invoke(expanded_query)
context = format_docs(retrieved_docs)

# Generate answer using original question
final_prompt = prompt.format(context=context, question=original_query)
answer = llm.invoke(final_prompt)

print(f"Question: {original_query}")
print(f"\nAnswer: {answer.content}")

Expanded: What are the primary anthropogenic and natural drivers of global warming, as identified in the latest available scientific evidence and assessed in accordance with the principles outlined in the IPCC's Framework for the Construction of Scenarios?

Question: What are the main causes of climate change?

Answer: The main cause of climate change is human influence, specifically the release of greenhouse gases (GHGs) through anthropogenic (human-induced) activities. This is stated in the following sentences:

"overwhelmingly due to human influence" (TS1.1 Context of a Changing Climate)
"human influence on the atmosphere, ocean, and land components of the climate system, taken together, is assessed as unequivocal for the first time in an IPCC assessment report" (Changes Across the Global Climate System)
"Earth system model simulations of the historical period since 1850 are only able to reproduce the observed changes in key climate indicators when anthropogenic forcings are include

In [28]:
# Pre-build BM25 indices once for all chunk sizes
# This avoids rebuilding inside the pipeline function every time

bm25_indices = {}
for size in [1000, 2500, 4000]:
    bm25_indices[size] = BM25Retriever.from_documents(
        all_chunk_sets[size],
        k=3
    )
    print(f"BM25 index built for chunk size {size}")

print("\nAll BM25 indices ready")

BM25 index built for chunk size 1000
BM25 index built for chunk size 2500
BM25 index built for chunk size 4000

All BM25 indices ready


In [29]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
import time

load_dotenv()

# Initialize LLM
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0
)

# RAG prompt
rag_prompt = PromptTemplate.from_template("""You are ClimateIQ, a climate science assistant.
Answer the question using ONLY the context provided below.
If the answer is not in the context, say "I don't have enough information to answer this."
Always mention which document your answer comes from.
If the question is simple, answer in plain language.
If the question is technical, use appropriate scientific language.

Context:
{context}

Question: {question}

Answer:""")

# Query expansion prompt
expansion_prompt = PromptTemplate.from_template("""You are a climate science expert familiar with IPCC reports.
Rewrite the following question using precise scientific terminology 
that would appear in IPCC assessment reports.
Return only the rewritten question, nothing else.

Original question: {question}

Rewritten question:""")

query_expander = expansion_prompt | llm | StrOutputParser()

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def run_rag_pipeline(query, retriever_type="dense", chunk_size=2500):
    """
    Run complete RAG pipeline with specified retriever type and chunk size.
    
    Args:
        query: User question
        retriever_type: "dense", "bm25", "hybrid", or "expansion"
        chunk_size: 1000, 2500, or 4000
    
    Returns:
        Dictionary with question, answer, retrieved chunks, and metadata
    """
    
    start_time = time.time()
    
    # Load correct vectorstore for chunk size
    collection_name = f"climate_chunks_{chunk_size}"
    vectorstore = Chroma(
        collection_name=collection_name,
        embedding_function=embeddings,
        persist_directory=chroma_path
    )
    
    # Build retriever based on type
    if retriever_type == "dense":
        retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
        search_query = query
        
    elif retriever_type == "bm25":
        retriever = bm25_indices[chunk_size]
        search_query = query
    
    elif retriever_type == "hybrid":
        dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
        retriever = EnsembleRetriever(
            retrievers=[bm25_indices[chunk_size], dense_retriever],
            weights=[0.5, 0.5]
        )
        search_query = query
        
   
        
    elif retriever_type == "expansion":
        retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
        search_query = query_expander.invoke({"question": query})
        
    else:
        raise ValueError(f"Unknown retriever type: {retriever_type}")
    
    # Retrieve chunks
    retrieved_docs = retriever.invoke(search_query)
    
    # Format context
    context = format_docs(retrieved_docs)
    
    # Generate answer
    final_prompt = rag_prompt.format(context=context, question=query)
    response = llm.invoke(final_prompt)
    answer = response.content
    
    end_time = time.time()
    
    # Build result dictionary
    result = {
        "question": query,
        "retriever_type": retriever_type,
        "chunk_size": chunk_size,
        "search_query": search_query,
        "answer": answer,
        "retrieved_chunks": [
            {
                "content": doc.page_content,
                "source": doc.metadata.get("source", "").split("\\")[-1],
                "page": doc.metadata.get("page", "")
            }
            for doc in retrieved_docs
        ],
        "num_chunks_retrieved": len(retrieved_docs),
        "time_taken": round(end_time - start_time, 2)
    }
    
    return result


# Test the function with all four retriever types
test_query = "What are the main causes of climate change?"

for retriever_type in ["dense", "bm25", "hybrid", "expansion"]:
    print(f"\n{'='*50}")
    print(f"Retriever: {retriever_type.upper()}")
    print(f"{'='*50}")
    result = run_rag_pipeline(test_query, retriever_type=retriever_type)
    print(f"Answer: {result['answer'][:600]}")
    print(f"Sources: {[c['source'] for c in result['retrieved_chunks']]}")
    print(f"Time: {result['time_taken']} seconds")


Retriever: DENSE
Answer: Human influence is the main driver of climate change, responsible for more than 50% of the change. This is stated in the context of the report, specifically in section A.3 and footnote 12.
Sources: ['IPCC_AR6_WGI_SPM.pdf', 'IPCC_AR6_WGII_TechnicalSummary.pdf', 'IPCC_AR6_WGI_TS.pdf']
Time: 0.8 seconds

Retriever: BM25
Answer: I don't have enough information to answer this.

(Note: The provided context does not explicitly mention the main causes of climate change. It discusses climate change information, mitigation pathways, and regional climate responses, but does not provide a detailed explanation of the causes of climate change.)
Sources: ['IPCC_AR6_WGIII_TechnicalSummary.pdf', 'IPCC_AR6_WGI_TS.pdf', 'IPCC_AR6_WGI_TS.pdf']
Time: 0.39 seconds

Retriever: HYBRID
Answer: Human influence is the main driver of climate change. This is stated in the context as "human-induced climate change is the main driver of these changes" (TS.5) and "human influence has very lik

In [1]:
import pickle
import os

# Save chunks to pickle file
chunks_path = r"C:\Users\Asus\Documents\Oulu Research Intern\ClimateIQ\chunks.pkl"

with open(chunks_path, "wb") as f:
    pickle.dump(all_chunk_sets, f)

print(f"Chunks saved successfully")
print(f"File size: {os.path.getsize(chunks_path) / 1024 / 1024:.1f} MB")

NameError: name 'all_chunk_sets' is not defined

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

chroma_path = r"C:\Users\Asus\Documents\Oulu Research Intern\ClimateIQ\Data\chroma_db"

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

# Extract chunks from ChromaDB collections
all_chunk_sets = {}

for size in [1000, 2500, 4000]:
    collection_name = f"climate_chunks_{size}"
    vectorstore = Chroma(
        collection_name=collection_name,
        embedding_function=embeddings,
        persist_directory=chroma_path
    )
    
    # Get all documents from collection
    data = vectorstore.get()
    
    from langchain_core.documents import Document
    chunks = [
        Document(
            page_content=doc,
            metadata=meta
        )
        for doc, meta in zip(data['documents'], data['metadatas'])
    ]
    
    all_chunk_sets[size] = chunks
    print(f"Loaded {len(chunks)} chunks for size {size}")

print("All chunks loaded from ChromaDB")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded 2200 chunks for size 1000
Loaded 1008 chunks for size 2500
Loaded 695 chunks for size 4000
All chunks loaded from ChromaDB


In [3]:
import pickle
import os

# Save chunks to pickle file
chunks_path = r"C:\Users\Asus\Documents\Oulu Research Intern\ClimateIQ\chunks.pkl"

with open(chunks_path, "wb") as f:
    pickle.dump(all_chunk_sets, f)

print(f"Chunks saved successfully")
print(f"File size: {os.path.getsize(chunks_path) / 1024 / 1024:.1f} MB")


Chunks saved successfully
File size: 6.9 MB


In [1]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
import re

def clean_document(text):
    if text is None:
        return ""
    text = re.sub(r'Figure\s+[A-Z]*\s*\d+\.\d+', '', text)
    text = re.sub(r'Box\s+[A-Z]*\s*\d+\.\d+', '', text)
    text = re.sub(r'\b[A-Z]{1,2}\.\d+\.\d+\b', '', text)
    text = re.sub(r'\{[^}]+\}', '', text)
    text = re.sub(r'^\d+$', '', text, flags=re.MULTILINE)
    text = re.sub(r'^[-_]+$', '', text, flags=re.MULTILINE)
    text = re.sub(r' +', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

# Load all chapter PDFs
print("Loading chapter PDFs...")
chapter_loader = DirectoryLoader(
    r"C:\Users\Asus\Documents\Oulu Research Intern\ClimateIQ\Data\chapters",
    glob="*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True
)

chapter_documents = list(chapter_loader.lazy_load())
print(f"\nTotal pages loaded from chapters: {len(chapter_documents)}")

# Clean all documents
for doc in chapter_documents:
    doc.page_content = clean_document(doc.page_content)

print("Cleaning done")

C:\Users\Asus\AppData\Local\Temp\ipykernel_19632\465154227.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader


Loading chapter PDFs...


100%|██████████| 32/32 [11:34<00:00, 21.70s/it]



Total pages loaded from chapters: 4018
Cleaning done


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chapter_chunk_configs = [
    {"size": 1000, "overlap": 100},
    {"size": 2500, "overlap": 250},
    {"size": 4000, "overlap": 400}
]

chapter_chunk_sets = {}

for config in chapter_chunk_configs:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config["size"],
        chunk_overlap=config["overlap"],
        length_function=len,
        separators=["\n\n", "\n", ".", " ", ""]
    )
    chunks = splitter.split_documents(chapter_documents)
    filtered = [c for c in chunks if len(c.page_content) >= 100]
    chapter_chunk_sets[config["size"]] = filtered
    print(f"Chunk size {config['size']}: {len(filtered)} chunks")

print("\nChunking complete")


Chunk size 1000: 28673 chunks
Chunk size 2500: 12532 chunks
Chunk size 4000: 8491 chunks

Chunking complete


In [3]:
# Verify chunking quality across all three sizes
for size in [1000, 2500, 4000]:
    print(f"\n{'='*50}")
    print(f"CHUNK SIZE {size}")
    print(f"{'='*50}")
    print(f"Total chunks: {len(chapter_chunk_sets[size])}")
    
    # Show 2 sample chunks
    sample1 = chapter_chunk_sets[size][100]
    sample2 = chapter_chunk_sets[size][500]
    
    print(f"\nSample 1:")
    print(f"Source: {sample1.metadata.get('source', '').split(chr(92))[-1]}")
    print(f"Page: {sample1.metadata.get('page', '')}")
    print(f"Length: {len(sample1.page_content)} characters")
    print(f"Content preview: {sample1.page_content[:200]}")
    
    print(f"\nSample 2:")
    print(f"Source: {sample2.metadata.get('source', '').split(chr(92))[-1]}")
    print(f"Page: {sample2.metadata.get('page', '')}")
    print(f"Length: {len(sample2.page_content)} characters")
    print(f"Content preview: {sample2.page_content[:200]}")

# Check for very short chunks
for size in [1000, 2500, 4000]:
    short_chunks = [c for c in chapter_chunk_sets[size] if len(c.page_content) < 200]
    print(f"\nChunk size {size}: {len(short_chunks)} chunks under 200 characters")


CHUNK SIZE 1000
Total chunks: 28673

Sample 1:
Source: IPCC_AR6_WGIII_Chapter02.pdf
Page: 16
Length: 937 characters
Content preview: a 50% probability as well as for limiting warming to 2°C with a 67% 
probability. Based on central estimates only, historical cumulative 
net CO2 emissions between 1850–2019 amount to about four fifth

Sample 2:
Source: IPCC_AR6_WGIII_Chapter02.pdf
Page: 76
Length: 981 characters
Content preview: potentials in the chemical sector. Energy, 153, 231–247, doi:10.1016/ 
j.energy. 2018.04.032.
Tanaka, K. and B.C. O’Neill, 2018: The Paris Agreement zero-emissions goal 
is not always consistent with 

CHUNK SIZE 2500
Total chunks: 12532

Sample 1:
Source: IPCC_AR6_WGIII_Chapter02.pdf
Page: 40
Length: 2056 characters
Content preview: particularly promising on both (medium confidence). Faster adoption 
and continued technological progress can play a  crucial role in 
accelerating the energy transition. However, the historical pace 

Sample 2:
Source: IPCC_AR6_WGI

In [4]:
# Refilter with stricter minimum
chapter_chunk_sets_clean = {}

for size in [1000, 2500, 4000]:
    original = len(chapter_chunk_sets[size])
    filtered = [c for c in chapter_chunk_sets[size] if len(c.page_content) >= 300]
    chapter_chunk_sets_clean[size] = filtered
    print(f"Chunk size {size}: {original} → {len(filtered)} chunks (removed {original - len(filtered)})")

# Use clean version going forward
chapter_chunk_sets = chapter_chunk_sets_clean
print("\nFiltering complete")


Chunk size 1000: 28673 → 27759 chunks (removed 914)
Chunk size 2500: 12532 → 12373 chunks (removed 159)
Chunk size 4000: 8491 → 8479 chunks (removed 12)

Filtering complete


In [5]:
import re

def is_reference_chunk(text):
    """Detect if a chunk is mostly references/citations"""
    # Count lines that look like citations (author, year pattern)
    lines = text.split('\n')
    citation_lines = sum(1 for line in lines 
                        if re.search(r'\d{4}[:\.]', line) 
                        and len(line) > 20)
    # If more than 40% of lines look like citations, it is a reference chunk
    return citation_lines / max(len(lines), 1) > 0.4

# Filter out reference chunks
chapter_chunk_sets_final = {}

for size in [1000, 2500, 4000]:
    original = len(chapter_chunk_sets[size])
    filtered = [c for c in chapter_chunk_sets[size] 
                if not is_reference_chunk(c.page_content)]
    chapter_chunk_sets_final[size] = filtered
    print(f"Chunk size {size}: {original} → {len(filtered)} chunks (removed {original - len(filtered)} reference chunks)")

chapter_chunk_sets = chapter_chunk_sets_final
print("\nReference filtering complete")

Chunk size 1000: 27759 → 20635 chunks (removed 7124 reference chunks)
Chunk size 2500: 12373 → 8784 chunks (removed 3589 reference chunks)
Chunk size 4000: 8479 → 5814 chunks (removed 2665 reference chunks)

Reference filtering complete


In [6]:
# Look at what was actually removed
removed_chunks = [c for c in chapter_chunk_sets_clean[1000] 
                  if is_reference_chunk(c.page_content)]

print(f"Inspecting 5 random removed chunks:\n")

import random
samples = random.sample(removed_chunks, 5)

for i, chunk in enumerate(samples):
    print(f"--- Removed Chunk {i+1} ---")
    print(f"Source: {chunk.metadata.get('source', '').split(chr(92))[-1]}")
    print(f"Page: {chunk.metadata.get('page', '')}")
    print(f"Content:\n{chunk.page_content[:400]}")
    print()
    

Inspecting 5 random removed chunks:

--- Removed Chunk 1 ---
Source: IPCC_AR6_WGII_Chapter05.pdf
Page: 188
Content:
projects: Development and adoption of improved varieties, creation of 
market-demand to benefit smallholder farmers and empowerment of 
national programmes in sub-Saharan Africa and South Asia. Plant Breed, 
138(4), 379–388, doi:10.1111/pbr.12744.
Varshney, R.K., et al., 2018: Can genomics deliver climate-change ready crops? 
Curr. Opin. Plant Biol., 45(Pt B), 205–211, doi:10.1016/j.pbi.2018.03.00

--- Removed Chunk 2 ---
Source: IPCC_AR6_WGI_Chapter03.pdf
Page: 104
Content:
Precipitation. Journal of Climate , 27(12), 4544–4565, doi: 10.1175/
jcli-d-13-00216.1.
Emile-Geay, J. et al., 2016: Links between tropical Pacific seasonal, interannual 
and orbital variability during the Holocene. Nature Geoscience , 9(2), 
168–173, doi:10.1038/ngeo2608.
England, M., A. Jahn, and L. Polvani, 2019: Nonuniform Contribution of 
Internal Variability to Recent Arctic Sea Ice Loss. Jo

--

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import os

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

chroma_path = r"C:\Users\Asus\Documents\Oulu Research Intern\ClimateIQ\Data\chroma_db"

print("Embeddings model loaded")
print(f"ChromaDB path: {chroma_path}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings model loaded
ChromaDB path: C:\Users\Asus\Documents\Oulu Research Intern\ClimateIQ\Data\chroma_db


In [9]:
from langchain_chroma import Chroma

for size in [1000, 2500, 4000]:
    print(f"Creating collection for chunk size {size}...")
    collection_name = f"climate_chapters_{size}"
    
    vectorstore = Chroma.from_documents(
        documents=chapter_chunk_sets[size],
        embedding=embeddings,
        collection_name=collection_name,
        persist_directory=chroma_path
    )
    
    print(f"Done — {len(chapter_chunk_sets[size])} chunks stored in {collection_name}")

print("\nAll chapter collections created successfully")

Creating collection for chunk size 1000...
Done — 20635 chunks stored in climate_chapters_1000
Creating collection for chunk size 2500...
Done — 8784 chunks stored in climate_chapters_2500
Creating collection for chunk size 4000...
Done — 5814 chunks stored in climate_chapters_4000

All chapter collections created successfully


In [11]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
import os

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

chroma_path = r"C:\Users\Asus\Documents\Oulu Research Intern\ClimateIQ\Data\chroma_db"

# Load SPM chunks from existing collections
print("Loading SPM chunks from ChromaDB...")
all_chunk_sets = {}
for size in [1000, 2500, 4000]:
    vectorstore = Chroma(
        collection_name=f"climate_chunks_{size}",
        embedding_function=embeddings,
        persist_directory=chroma_path
    )
    data = vectorstore.get()
    chunks = [
        Document(page_content=doc, metadata=meta)
        for doc, meta in zip(data['documents'], data['metadatas'])
    ]
    all_chunk_sets[size] = chunks
    print(f"SPM chunk size {size}: {len(chunks)} chunks loaded")

# Load chapter chunks from existing collections
print("\nLoading chapter chunks from ChromaDB...")
chapter_chunk_sets = {}
for size in [1000, 2500, 4000]:
    vectorstore = Chroma(
        collection_name=f"climate_chapters_{size}",
        embedding_function=embeddings,
        persist_directory=chroma_path
    )
    data = vectorstore.get()
    chunks = [
        Document(page_content=doc, metadata=meta)
        for doc, meta in zip(data['documents'], data['metadatas'])
    ]
    chapter_chunk_sets[size] = chunks
    print(f"Chapter chunk size {size}: {len(chunks)} chunks loaded")

print("\nAll chunks reloaded successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading SPM chunks from ChromaDB...
SPM chunk size 1000: 2200 chunks loaded
SPM chunk size 2500: 1008 chunks loaded
SPM chunk size 4000: 695 chunks loaded

Loading chapter chunks from ChromaDB...
Chapter chunk size 1000: 20635 chunks loaded
Chapter chunk size 2500: 8784 chunks loaded
Chapter chunk size 4000: 5814 chunks loaded

All chunks reloaded successfully


In [12]:
# Combine SPM chunks and chapter chunks into unified full corpus
print("Combining SPM and chapter chunks...")

combined_chunk_sets = {}

for size in [1000, 2500, 4000]:
    spm_chunks = all_chunk_sets[size]  # your existing SPM chunks
    chapter_chunks = chapter_chunk_sets[size]  # your new chapter chunks
    combined = spm_chunks + chapter_chunks
    combined_chunk_sets[size] = combined
    print(f"Chunk size {size}: {len(spm_chunks)} SPM + {len(chapter_chunks)} chapters = {len(combined)} total")

print("\nCombination complete")


Combining SPM and chapter chunks...
Chunk size 1000: 2200 SPM + 20635 chapters = 22835 total
Chunk size 2500: 1008 SPM + 8784 chapters = 9792 total
Chunk size 4000: 695 SPM + 5814 chapters = 6509 total

Combination complete


In [13]:
# Create unified full corpus collections
print("Creating unified full corpus collections...")

for size in [1000, 2500, 4000]:
    print(f"\nCreating climate_full_{size}...")
    collection_name = f"climate_full_{size}"
    
    vectorstore = Chroma.from_documents(
        documents=combined_chunk_sets[size],
        embedding=embeddings,
        collection_name=collection_name,
        persist_directory=chroma_path
    )
    
    print(f"Done — {len(combined_chunk_sets[size])} chunks stored in {collection_name}")

print("\nAll unified collections created successfully")

Creating unified full corpus collections...

Creating climate_full_1000...
Done — 22835 chunks stored in climate_full_1000

Creating climate_full_2500...
Done — 9792 chunks stored in climate_full_2500

Creating climate_full_4000...
Done — 6509 chunks stored in climate_full_4000

All unified collections created successfully


In [14]:
import pickle

# Save full corpus chunks to pickle
chunks_path = r"C:\Users\Asus\Documents\Oulu Research Intern\ClimateIQ\chunks.pkl"

with open(chunks_path, "wb") as f:
    pickle.dump(combined_chunk_sets, f)

import os
print(f"Chunks saved successfully")
print(f"File size: {os.path.getsize(chunks_path) / 1024 / 1024:.1f} MB")


Chunks saved successfully
File size: 74.0 MB


In [15]:
import pickle
import os

# Save only the 2500 chunk size — this is what the app uses
chunks_path = r"C:\Users\Asus\Documents\Oulu Research Intern\ClimateIQ\chunks.pkl"

with open(chunks_path, "wb") as f:
    pickle.dump(combined_chunk_sets[2500], f)

print(f"Chunks saved successfully")
print(f"File size: {os.path.getsize(chunks_path) / 1024 / 1024:.1f} MB")


Chunks saved successfully
File size: 23.3 MB
